# Read an operating-point optimization record

Level: 1. Prerequisites: Level 2 of comminution and level 1 of learning. Reading time: 18 minutes.

## Objectives

Separate the operating decisions, objective, feasibility constraints and measured search effort in the committed optimizer result. Explain why an optimum is conditional on the authored case and its contract.

## Setup

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0
```

The OreFlow pipeline is imported from `data-pipeline/`; this notebook never installs packages.

Configure imports and record versions. This makes the method run reproducible.

In [1]:
# Import analysis tools and record their exact versions.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve the local pipeline from the repository-relative layout.
repo = Path('../../../').resolve()
sys.path.insert(0, str(repo / 'data-pipeline'))
for package in ('numpy', 'scipy', 'pandas', 'matplotlib', 'nbclient', 'nbformat', 'ipykernel'):
    print(package, importlib.metadata.version(package))

numpy 2.2.6
scipy 1.17.1
pandas 2.2.3
matplotlib 3.11.2
nbclient 0.11.0
nbformat 5.11.1
ipykernel 7.4.0


## Data

Owner: OreFlow authors; `data/derived/benchmark.json` and `data/derived/contract/operating_contract.json`, MIT project licence. The optimum is calculated for an authored scenario, not a plant. Repository: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Method

OreFlow maximizes recovered primary metal relative to the base point while enforcing concentrate-grade, installed-power and water limits. A generalized pattern search works in the unit cube of declared decision bounds. A candidate rejected by the contract or engine is never treated as feasible. The committed record exposes the decisions, active constraints, evaluations and weight path.

Load the exact benchmark row and display the base-to-optimum decisions with their feasibility evidence.

In [2]:
import json
from pathlib import Path
import pandas as pd
from pipeline.cases.catalog import CASES

# Read the committed optimizer record for one authored rougher case.
repo = Path('../../../').resolve()
benchmark = json.loads((repo / 'data/derived/benchmark.json').read_text(encoding='utf-8'))
record = benchmark['optimization']['copper_porphyry_soft']['nominal']
case = next(item for item in CASES if item.id == 'copper_porphyry_soft')
case_record = json.loads((repo / 'data/derived/cases/copper_porphyry_soft.json').read_text(encoding='utf-8'))
base_point = case_record['variants'][0]['point']
print('status:', record['status'], '| algorithm: generalized pattern search')
print('base feasible:', record['base_feasible'], '| screened:', record['screened'], '| evaluations:', record['evaluations'])
rows = [{'state': 'base', **{name: base_point[name] for name in record['decisions']}}, {'state': 'optimum', **record['decisions']} ]
display(pd.DataFrame(rows).set_index('state').T)

status: optimal | algorithm: generalized pattern search
base feasible: True | screened: True | evaluations: 309


state,base,optimum
target_p80_um,150.0,124.291992
collector_gpt,25.0,74.951172
jg_cm_s,1.4,2.499609


Compare the objective improvement with the active physical constraint. The optimizer's objective does not include commodity price or operating cost.

In [3]:
# Compare the objective improvement with the active physical constraint.
print('recovered metal gain (%):', round(record['gain_pct'], 3))
print('active constraints:', record['active'])
print('same optimum without surrogate screen:', record['same_optimum_without_screen'])
print('screened engine evaluations:', record['evaluations'])
print('unscreened engine evaluations:', record['evaluations_without_screen'])
print('contract decision bounds:')
inputs = json.loads((repo / 'data/derived/contract/operating_contract.json').read_text(encoding='utf-8'))['cases']['copper_porphyry_soft']['inputs']
display(pd.DataFrame([{'decision': name, 'min': inputs[name]['min'], 'max': inputs[name]['max'], 'unit': inputs[name]['unit']} for name in record['decisions']]).set_index('decision'))

recovered metal gain (%): 5.744
active constraints: ['power']
same optimum without surrogate screen: True
screened engine evaluations: 309
unscreened engine evaluations: 266
contract decision bounds:


,min,max,unit
decision,,,
target_p80_um,75.0,300.0,um
collector_gpt,0.0,75.0,g/t
jg_cm_s,0.5,2.5,cm/s


## Results

The record reports a feasible candidate with a measured gain under the declared objective and constraints. The same optimum with and without screening is a useful parity check; it does not establish a global optimum or economic value. All inputs and plant capacities are scenario assumptions.

## Summary

You read the optimizer as a constrained engineering calculation: the result depends on the objective, bounds, constraints and starting case. A feasible result is not an operating recommendation.

## References

- https://doi.org/10.1137/S1052623493250780
- https://doi.org/10.1137/070692662